In [1]:
# import Python packages
import pandas as pd
import numpy as np
from numpy import array
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse
import scipy
import scipy.stats as ss
from skbio.stats.distance import permanova
from skbio.stats.ordination import OrdinationResults
import biom
from biom import load_table
from gemelli.rpca import rpca
from matplotlib.patches import Ellipse
from skbio.stats.distance import permanova, DistanceMatrix
from itertools import combinations
import qiime2 as q2
from qiime2 import Artifact
from qiime2 import Metadata
import os
from biom import Table

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message="You passed a edgecolor/edgecolors .*"
)

# Set display precision globally
pd.set_option('display.float_format', '{:.10f}'.format)

In [2]:
# ----------------------------------------------------------------------------
# Copyright (c) 2016-2023, QIIME 2 development team.
#
# Distributed under the terms of the Modified BSD License.
#
# The full license is in the file LICENSE, distributed with this software.
# ----------------------------------------------------------------------------

# Note: Functions are from https://github.com/qiime2/q2-feature-table/blob/dev/q2_feature_table/_filter.py; an alternative to direct import from my qiime2 environment
def _validate_nonempty_table(table):
    if table.is_empty():
        raise ValueError("The resulting table is empty. This can happen if "
                         "you filter all samples or features out of the "
                         "table. Please check your filtering parameters and "
                         "try again.")


def _get_biom_filter_function(ids_to_keep, min_frequency, max_frequency,
                              min_nonzero, max_nonzero):
    ids_to_keep = set(ids_to_keep)
    if max_frequency is None:
        max_frequency = np.inf
    if max_nonzero is None:
        max_nonzero = np.inf

    def f(data_vector, id_, metadata):
        return (id_ in ids_to_keep) and \
               (min_frequency <= data_vector.sum() <= max_frequency) and \
               (min_nonzero <= (data_vector > 0).sum() <= max_nonzero)
    return f


_other_axis_map = {'sample': 'observation', 'observation': 'sample'}


def _filter_table(table, min_frequency, max_frequency, min_nonzero,
                  max_nonzero, metadata, where, axis, exclude_ids=False,
                  filter_opposite_axis=True,
                  allow_empty_table=False):
    if min_frequency == 0 and max_frequency is None and min_nonzero == 0 and\
       max_nonzero is None and metadata is None and where is None and\
       exclude_ids is False:
        raise ValueError("No filtering was requested.")
    if metadata is None and where is not None:
        raise ValueError("Metadata must be provided if 'where' is "
                         "specified.")
    if metadata is None and exclude_ids is True:
        raise ValueError("Metadata must be provided if 'exclude_ids' "
                         "is True.")
    if metadata is not None:
        ids_to_keep = metadata.get_ids(where=where)
    else:
        ids_to_keep = table.ids(axis=axis)
    if exclude_ids is True:
        ids_to_keep = set(table.ids(axis=axis)) - set(ids_to_keep)

    filter_fn1 = _get_biom_filter_function(
        ids_to_keep, min_frequency, max_frequency, min_nonzero, max_nonzero)
    table.filter(filter_fn1, axis=axis, inplace=True)

    # filter on the opposite axis to remove any entities that now have a
    # frequency of zero
    if filter_opposite_axis:
        table.remove_empty(axis=_other_axis_map[axis], inplace=True)

    if not allow_empty_table:
        _validate_nonempty_table(table)


def filter_samples(table: biom.Table, min_frequency: int = 0,
                   max_frequency: int = None, min_features: int = 0,
                   max_features: int = None,
                   metadata: q2.Metadata = None, where: str = None,
                   exclude_ids: bool = False,
                   filter_empty_features: bool = True,
                   allow_empty_table: bool = False)\
                  -> biom.Table:
    _filter_table(table=table, min_frequency=min_frequency,
                  max_frequency=max_frequency, min_nonzero=min_features,
                  max_nonzero=max_features, metadata=metadata,
                  where=where, axis='sample', exclude_ids=exclude_ids,
                  filter_opposite_axis=filter_empty_features,
                  allow_empty_table=allow_empty_table
                  )
    return table

In [16]:
# Function to draw an ellipse based on the covariance of the points
def draw_ellipse(mean, cov, ax, color, label=None, scale_factor=1.5):
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    order = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    
    # Calculate the angle between the x-axis and the largest eigenvector
    angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
    
    # Width and height of the ellipse (scaling applied)
    width, height = 2 * np.sqrt(eigenvalues) * scale_factor
    
    # Create the ellipse and add it to the plot
    ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
    ax.add_patch(ell)

In [5]:
#not rarefied ref hit table
v4_table_Shiseido = Artifact.load('../data/amplicon/130729_feature-table_refhit.qza')
#v4_table_Shiseido = Artifact.load('../data/217010_rarefied_table_filtered.qza')


#not filtered metabolomics
metabolomics_table_Shiseido = Artifact.load('../data/metabolomics/Metabolomics_data_sample_raw_05162025.qza')


metadata_Shiseido = pd.read_csv('../data/Shiseido_metadata_ctf.tsv', sep='\t', index_col=0)


#for the not rarefied table and the unfiltered metabolomics table
metadata_Shiseido.index = metadata_Shiseido.index.str.replace('^130728\\.', '', regex=True)


In [6]:
# Define bins and labels
bins = [float('-inf'), 3.5, 16.875, float('inf')]
labels = ['low', 'medium', 'high']

# Create the 'sebum_group' column
metadata_Shiseido['sebum_group'] = pd.cut(metadata_Shiseido['sebum'], bins=bins, labels=labels, right=True)

In [7]:
print(metadata_Shiseido['sebum_group'])
print(metadata_Shiseido['sebum_group'].value_counts())

#SampleID
11369.m01.01A    medium
11369.m01.01B    medium
11369.m01.02A    medium
11369.m01.02B    medium
11369.m01.03A    medium
                  ...  
11369.p60.3A     medium
11369.p60.3B     medium
11369.p61.1A        NaN
11369.p61.2A        NaN
11369.p61.3A        NaN
Name: sebum_group, Length: 483, dtype: category
Categories (3, object): ['low' < 'medium' < 'high']
medium    230
low       130
high      120
Name: sebum_group, dtype: int64


In [8]:
metadata_Shiseido['Face_site']

#SampleID
11369.m01.01A    Forehead
11369.m01.01B    Forehead
11369.m01.02A    Forehead
11369.m01.02B    Forehead
11369.m01.03A        Nose
                   ...   
11369.p60.3A        Cheek
11369.p60.3B        Cheek
11369.p61.1A     Forehead
11369.p61.2A         Nose
11369.p61.3A        Cheek
Name: Face_site, Length: 483, dtype: object

In [9]:
table_sample_ids = v4_table_Shiseido.view(pd.DataFrame).index

metadata_Shiseido_filtered = metadata_Shiseido.loc[
    metadata_Shiseido.index.intersection(table_sample_ids)
]

print(metadata_Shiseido_filtered.head())


metadata_Shiseido_filtered.index.name = '#SampleID'

q2_meta_v4 = Metadata(
    metadata_Shiseido_filtered.applymap(lambda x: str(x) if isinstance(x, bool) else x)
)


                  SampleID_2 Sample_name_old  Line_No. PlateNo WellPosition  \
11369.m01.01A  11369.m01.01A         m01-01A         1     m01          a01   
11369.m01.01B  11369.m01.01B         m01-01B        13     m01          b01   
11369.m01.02A  11369.m01.02A         m01-02A         2     m01          a02   
11369.m01.02B  11369.m01.02B         m01-02B        14     m01          b02   
11369.m01.03A  11369.m01.03A         m01-03A         3     m01          a03   

              SmplSite  SubjectNo microbiome_ID Timepoint  \
11369.m01.01A      01A        NaN        m01-01    Time 0   
11369.m01.01B      01B        NaN        m01-01    Time 0   
11369.m01.02A      02A        NaN        m01-02    Time 0   
11369.m01.02B      02B        NaN        m01-02    Time 0   
11369.m01.03A      03A        NaN        m01-03    Time 0   

                                                TITLE  ...  \
11369.m01.01A  Japanese skin mirobiome and metabolome  ...   
11369.m01.01B  Japanese skin mirob

In [10]:
table_sample_ids = metabolomics_table_Shiseido.view(pd.DataFrame).index

metadata_Shiseido_filtered = metadata_Shiseido.loc[
    metadata_Shiseido.index.intersection(table_sample_ids)
]

print(metadata_Shiseido_filtered.head())


metadata_Shiseido_filtered.index.name = '#SampleID'
q2_meta_metabolomics = Metadata(metadata_Shiseido_filtered.applymap(lambda x: str(x) if isinstance(x, bool) else x))

                  SampleID_2 Sample_name_old  Line_No. PlateNo WellPosition  \
11369.m01.01A  11369.m01.01A         m01-01A         1     m01          a01   
11369.m01.01B  11369.m01.01B         m01-01B        13     m01          b01   
11369.m01.02A  11369.m01.02A         m01-02A         2     m01          a02   
11369.m01.02B  11369.m01.02B         m01-02B        14     m01          b02   
11369.m01.03A  11369.m01.03A         m01-03A         3     m01          a03   

              SmplSite  SubjectNo microbiome_ID Timepoint  \
11369.m01.01A      01A        NaN        m01-01    Time 0   
11369.m01.01B      01B        NaN        m01-01    Time 0   
11369.m01.02A      02A        NaN        m01-02    Time 0   
11369.m01.02B      02B        NaN        m01-02    Time 0   
11369.m01.03A      03A        NaN        m01-03    Time 0   

                                                TITLE  ...  \
11369.m01.01A  Japanese skin mirobiome and metabolome  ...   
11369.m01.01B  Japanese skin mirob

In [11]:
group_order = ['low', 'medium', 'high']
short_group_names = {"medium": "M", "low": "L", "high": "H", "NA": "NaN"}

In [12]:
from qiime2 import Artifact, Metadata
from qiime2.plugins.feature_table.methods import filter_samples

# Step 1: Filter metadata for valid face sites and remove "na" Subject_IDs
valid_sites = ['Forehead', 'Nose', 'Cheek']
initial_sample_count = metadata_Shiseido.shape[0]

filtered_metadata = metadata_Shiseido[
    (metadata_Shiseido['Face_site'].isin(valid_sites)) &
    (metadata_Shiseido['Subject_ID'].str.lower() != "na")
]

filtered_sample_count = filtered_metadata.shape[0]
print(f"Samples before filtering: {initial_sample_count}")
print(f"Samples after filtering (Face_site + Subject_ID): {filtered_sample_count}")

# Step 2: Convert all categorical columns to string (object)
filtered_metadata = filtered_metadata.apply(
    lambda col: col.astype(str) if col.dtype.name == 'category' else col
)

# Step 3: Convert to QIIME2 Metadata
filtered_metadata_q2 = Metadata(filtered_metadata)

# Step 4: Filter v4 table
v4_table_filtered = filter_samples(
    table=v4_table_Shiseido,
    metadata=filtered_metadata_q2
).filtered_table

# Step 5: Filter metabolomics table
metabolomics_table_filtered = filter_samples(
    table=metabolomics_table_Shiseido,
    metadata=filtered_metadata_q2
).filtered_table

# Step 6: Check number of samples retained in the filtered tables
v4_df = v4_table_filtered.view(pd.DataFrame)
metab_df = metabolomics_table_filtered.view(pd.DataFrame)

print(f"v4_table_filtered: {v4_df.shape[1]} samples, {v4_df.shape[0]} features")
print(f"metabolomics_table_filtered: {metab_df.shape[1]} samples, {metab_df.shape[0]} features")


Samples before filtering: 483
Samples after filtering (Face_site + Subject_ID): 435
v4_table_filtered: 18396 samples, 435 features
metabolomics_table_filtered: 1987 samples, 432 features


In [13]:
for sid in metadata_Shiseido_filtered['Subject_ID']:
    print(sid)


m01
m01
m01
m01
m01
m01
m01
m01
m01
m01
m01
m01
m01
m01
m01
m01
m01
m01
m01
m01
m01
m01
m01
m01
m01
m01
m01
m01
m01
m01
m02
m02
m02
m02
m02
m02
m02
m02
m02
m02
m02
m02
m02
m02
m02
m02
m02
m02
m02
m02
m02
m02
m02
m02
m02
m02
m02
m02
m02
m03
m03
m03
m03
m03
m03
m03
m03
m03
m03
m03
m03
m03
m03
m03
m03
m03
m03
m03
m03
m03
m03
m03
m03
m03
m03
m03
m03
m03
m03
m04
m04
m04
m04
m04
m04
m04
m04
m04
m04
m04
m04
m04
m04
m04
m04
m04
m04
m04
m04
m04
m04
m04
m04
m04
m04
m04
m04
m04
p01
p01
p01
p01
p01
p01
p02
p02
p02
p02
p02
p02
p03
p03
p03
p03
p03
p03
p04
p04
p04
p04
p04
p04
p05
p05
p05
p05
p05
p05
p06
p06
p06
p06
p06
p06
p07
p07
p07
p07
p07
p07
p08
p08
p08
p08
p08
p08
p09
p09
p09
p09
p09
p09
p10
p10
p10
p10
p10
p10
p11
p11
p11
p11
p11
p11
p12
p12
p12
p12
p12
p12
p13
p13
p13
p13
p13
p13
p14
p14
p14
p14
p14
p14
p15
p15
p15
p15
p15
p15
p16
p16
p16
p16
p16
p16
p17
p17
p17
p17
p17
p17
p18
p18
p18
p18
p18
p18
p19
p19
p19
p19
p19
p19
p20
p20
p20
p20
p20
p20
p21
p21
p21
p21
p21
p21
p22
p22
p22
p22
p22
p22


In [14]:
tables_metadata = [
    (v4_table_filtered, q2_meta_v4, '16S rRNA (V4) RPCA Beta Diversity', 'v4_130729', '../figures/main')]


# Update colors and group name mapping for new labels
group_colors = {
    "low": "#4575b4",
    "medium": "#fee0b6",
    "high": "#e08214",
    "NaN":"grey"
}

group_name_mapping = {
    "low": "Low (L, <3.5 µg/cm²)",
    "medium": "Medium (M, 3.5-16.9 µg/cm²)",
    "high": "High (H, >16.9 µg/cm²)",
    "NaN": "NaN"
}

In [17]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from skbio import DistanceMatrix
from skbio.stats.distance import permanova
from scipy.stats import f_oneway
from skbio.stats.distance import permdisp

# Main loop
for table, metadata, plot_title, file_suffix, output_path in tables_metadata:
    print(plot_title)
    os.makedirs(output_path, exist_ok=True)

    np.seterr(divide='ignore')
    table_biom = table.view(Table)
    ordination_artifact, distance = rpca(table_biom)
    ordination = ordination_artifact

    # Ordination + metadata
    spca_df = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    spca_df.columns = ['PC1', 'PC2', 'PC3']
    metadata_df = metadata.to_dataframe()
    spca_df["Group"] = spca_df.index.map(metadata_df["sebum_group"])
    spca_df = spca_df.dropna(subset=["Group"])

    # Prepare distance matrix and filtered metadata
    common_ids = set(metadata_df.index).intersection(distance.ids)
    metadata_df = metadata_df.loc[list(common_ids)].dropna(subset=['sebum_group'])
    distance_matrix = DistanceMatrix(distance.to_data_frame().values, ids=distance.ids).filter(metadata_df.index, strict=True)

    # PERMANOVA
    results = []
    for group1, group2 in combinations(metadata_df["sebum_group"].unique(), 2):
        subset_ids = metadata_df[metadata_df["sebum_group"].isin([group1, group2])].index
        subset_distance = distance_matrix.filter(subset_ids, strict=True)
        subset_metadata = metadata_df.loc[subset_ids]
        result = permanova(subset_distance, subset_metadata["sebum_group"])
        results.append({'Group1': group1, 'Group2': group2, 'Test Statistic': result['test statistic'], 'p-value': result['p-value']})
    results_df = pd.DataFrame(results)

    # --- Compute centroid per group ---
    group_centroids = spca_df.groupby("Group")[["PC1", "PC2"]].mean()

    def compute_centroid_distance(row):
        group = row["Group"]
        centroid = group_centroids.loc[group]
        return np.linalg.norm(row[["PC1", "PC2"]].values - centroid.values)

    spca_df["distance_to_centroid"] = spca_df.apply(compute_centroid_distance, axis=1)

    # PERMDISP
    group_labels = spca_df["Group"].reindex(distance_matrix.ids)
    permdisp_result = permdisp(distance_matrix, group_labels, permutations=999)
    f_stat = permdisp_result['test statistic']
    p_disp = permdisp_result['p-value']

    print(f"PERMDISP result: F = {f_stat:.4f}, p = {p_disp:.4e}")

    # === RPCA Scatter Plot ===
    fig, ax = plt.subplots(figsize=(7, 8))

    legend_labels_with_n = {}
    for group in group_order:
        if group in spca_df["Group"].values:
            data = spca_df[spca_df["Group"] == group]
            group_n = data.shape[0]
            label = f"{group_name_mapping[group]} (n = {group_n})"
            legend_labels_with_n[group] = label
            sns.scatterplot(
                data=data, x="PC1", y="PC2",
                facecolor=group_colors[group],
                edgecolor='k', alpha=0.9,
                linewidth=0.5, s=100, ax=ax,
                label=label
            )

    for label, data in spca_df.groupby("Group"):
        points = data[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        if len(data) > 2:
            cov = np.cov(points, rowvar=False)
            draw_ellipse(mean, cov, ax, group_colors[label], scale_factor=2)
        ax.scatter(mean[0], mean[1], color=group_colors[label], marker=(8, 2, 0), s=300, edgecolor='black', zorder=5, linewidth=1)

    ax.set_aspect('equal')
    ax.set_xlabel(f'PC1 ({ordination.proportion_explained[0] * 100:.2f}%)', fontsize=18)
    ax.set_ylabel(f'PC2 ({ordination.proportion_explained[1] * 100:.2f}%)', fontsize=18)
    ax.text(0.5, 1.05, plot_title, fontsize=20, va='top', ha='center', transform=ax.transAxes)
    ax.set_xlim(-0.15, 0.15)
    ax.set_ylim(-0.17, 0.17)
    ticks = [-0.1, 0.0, 0.1]
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.set_xticklabels(ticks, fontsize=18)
    ax.set_yticklabels(ticks, fontsize=18)
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

    # Legend
    desired_order = [legend_labels_with_n[g] for g in group_order if g in legend_labels_with_n]
    handles, labels = ax.get_legend_handles_labels()
    label_to_handle = dict(zip(labels, handles))
    ordered_handles = [label_to_handle[l] for l in desired_order if l in label_to_handle]
    ordered_labels = [l for l in desired_order if l in label_to_handle]
    legend = ax.legend(
        ordered_handles, ordered_labels,
        frameon=False, fontsize=14, title_fontsize=16,
        loc="upper left", bbox_to_anchor=(0, 1.0),
        borderaxespad=0.2, title='Sebum Group'
    )
    legend.get_title().set_ha('left')

    # PERMANOVA box text
    #pvalue_text = '\n'.join(
    #    f"{short_group_names[r['Group1']]} vs {short_group_names[r['Group2']]}: "
    #    f"{'*** ' if r['p-value'] <= 0.001 else '** ' if 0.001 < r['p-value'] <= 0.01 else '* ' if r['p-value'] <= 0.05 else ''}"
    #    f"p={r['p-value']:.3f}; F-stat={r['Test Statistic']:.2f}"
    #    for _, r in results_df.iterrows()
    #)


    pvalue_text = '\n'.join(
        f"{short_group_names[r['Group1']]} vs {short_group_names[r['Group2']]}: "
        f"p={r['p-value']:.3f} "
        f"{'***' if r['p-value'] <= 0.001 else '**' if r['p-value'] <= 0.01 else '*' if r['p-value'] <= 0.05 else ''} "
        f"F={r['Test Statistic']:.2f}"
        for _, r in results_df.iterrows()
    )



    ax.text(ax.get_xlim()[0] + 0.01, ax.get_ylim()[0] + 0.01, pvalue_text,
            fontsize=14, va='bottom', ha='left',
            bbox=dict(boxstyle="round,pad=0.3", edgecolor="gray", facecolor="white", alpha=0.8))

    plt.tight_layout()
    plt.savefig(os.path.join(output_path, f"{file_suffix}_rPCA_Forehead_Nose_Cheek_02022026.png"), dpi=600)
    #plt.savefig(os.path.join(output_path, f"{file_suffix}_rPCA_Forehead_Nose_Cheek_06302025.png"), dpi=600)
    #plt.savefig(os.path.join(output_path, f"{file_suffix}_rPCA_06302025.png"), dpi=600)
    plt.close(fig)
    print(f"RPCA plot saved to {output_path}")

    # === Beta Dispersion Plot ===
    group_display_mapping = {
        "low": "Low \n (< 3.5 µg/cm²)",
        "medium": "Medium \n (3.5 – 16.9 µg/cm²)",
        "high": "High \n (> 16.9 µg/cm²)"
    }

    # Create group display with (n = X)

    group_display_with_n = {
        g: f"{group_display_mapping[g]}\n(n = {spca_df[spca_df['Group'] == g].shape[0]})"
        for g in group_order if g in spca_df["Group"].values
    }

    spca_df["Group_Display"] = spca_df["Group"].map(group_display_with_n)
    group_display_order = [group_display_with_n[g] for g in group_order if g in group_display_with_n]

    group_colors_display = {
        group_display_with_n[k]: group_colors[k]
        for k in group_order if k in group_display_with_n
    }

    plt.figure(figsize=(6, 4))
    sns.boxplot(
        data=spca_df,
        x="Group_Display",
        y="distance_to_centroid",
        order=group_display_order,
        palette=group_colors_display
    )

    # Overlay colored dots per group
    for group in group_order:
        if group in spca_df["Group"].values:
            display_label = group_display_with_n[group]
            group_data = spca_df[spca_df["Group"] == group]
            sns.stripplot(
                data=group_data,
                x="Group_Display",
                y="distance_to_centroid",
                order=group_display_order,
                color=group_colors[group],
                size=5,
                alpha=0.7
            )

    plt.ylabel("Distance to Centroid (RPCA)")
    plt.xlabel("")
    plt.title(f"Beta Dispersion by Sebum Group\nPERMDISP p = {p_disp:.4e}; F-stat = {f_stat:.4f}")
    plt.tight_layout()
    plt.savefig(os.path.join(output_path, f"{file_suffix}_beta_dispersion_Forehead_Nose_Cheek_02022026.png"), dpi=600)
    #plt.savefig(os.path.join(output_path, f"{file_suffix}_beta_dispersion_Forehead_Nose_Cheek_06302025.png"), dpi=600)
    #plt.savefig(os.path.join(output_path, f"{file_suffix}_beta_dispersion_06302025.png"), dpi=600)
    plt.close()
    print(f"Beta dispersion plot saved to {output_path}")


16S rRNA (V4) RPCA Beta Diversity
PERMDISP result: F = 6.1010, p = 1.0000e-03


/var/folders/v_/f2630xts0zgcr95tld13kd4c0000gn/T/ipykernel_52129/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/v_/f2630xts0zgcr95tld13kd4c0000gn/T/ipykernel_52129/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/v_/f2630xts0zgcr95tld13kd4c0000gn/T/ipykernel_52129/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will becom

RPCA plot saved to ../figures/main
Beta dispersion plot saved to ../figures/main
